# Data Pipeline: Transcribe, Translate, Synthesize, Align

**GPU Budget:** ~6-8 hours on Kaggle T4 (per direction)

This notebook runs the complete data preparation pipeline for creating
synthetic parallel speech pairs:

```
Source Audio  -->  Whisper (timestamps + text)  -->  MADLAD translate
    -->  MMS-TTS synthesize target speech  -->  Contextual align
    -->  Silence insertion  -->  Save to Kaggle working dir
```

### Data Sources
- **Sw->En**: KenSpeech Swahili (auto-downloaded from HuggingFace)
- **En->Sw**: FLEURS en_us (auto-downloaded from HuggingFace)
- No manual dataset download required

### Run Order
Run this notebook **twice** -- once per direction:
1. First run: **Sw->En** (source=sw, target=en) -- KenSpeech source, pretrained `facebook/mms-tts-eng`
2. Second run: **En->Sw** (source=en, target=sw) -- FLEURS source, pretrained `facebook/mms-tts-swh`

**After each run:** save `/kaggle/working/hibiki-sw` output as a new Kaggle Dataset version before the session ends.

In [ ]:
# Install dependencies
!pip install -q transformers accelerate torchaudio soundfile scipy faster-whisper sentencepiece

In [ ]:
import os
import sys
import torch

REPO_DIR = '/kaggle/input/datasets/victormugambi/hibiki-sw/hibiki-sw'
sys.path.insert(0, REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR  # inherited by all !python subprocess calls

BASE_DIR = '/kaggle/working/hibiki-sw'
os.makedirs(BASE_DIR, exist_ok=True)

# === CHANGE THESE FOR EACH RUN ===
# Run 1: SOURCE_LANG='sw', TARGET_LANG='en'
# Run 2: SOURCE_LANG='en', TARGET_LANG='sw'
SOURCE_LANG = 'sw'
TARGET_LANG = 'en'

# Data source: 'kenspeech' for Sw->En, 'directory' for En->Sw (FLEURS)
SOURCE = 'kenspeech' if SOURCE_LANG == 'sw' else 'directory'

MAX_SAMPLES = 10000  # KenSpeech has ~5816, FLEURS en_us has ~2009

print(f'Direction: {SOURCE_LANG} -> {TARGET_LANG}')
print(f'Source: {SOURCE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# Verify data source accessibility
if SOURCE == 'kenspeech':
    # KenSpeech auto-downloads from HuggingFace
    from data.prepare.kenspeech_loader import KenSpeechLoader
    ks = KenSpeechLoader(load_audio=False)
    print(f'KenSpeech loaded: {len(ks)} samples')
    stats = ks.get_stats()
    print(f'Unique speakers: {stats["unique_speakers"]}')
    print(f'Avg sentence length: {stats["avg_sentence_length"]:.0f} chars')
    del ks
    print('Dataset ready!')

elif SOURCE == 'directory':
    # For En->Sw: extract FLEURS en_us audio to a temp directory
    FLEURS_AUDIO_DIR = f'{BASE_DIR}/fleurs_en_audio'
    if not os.path.exists(FLEURS_AUDIO_DIR) or len(os.listdir(FLEURS_AUDIO_DIR)) == 0:
        print('Extracting FLEURS en_us audio to WAV files...')
        os.makedirs(FLEURS_AUDIO_DIR, exist_ok=True)
        from datasets import load_dataset
        import soundfile as sf
        import numpy as np

        ds = load_dataset("google/fleurs", "en_us", split="train")
        for i, sample in enumerate(ds):
            audio = sample["audio"]
            wav_path = os.path.join(FLEURS_AUDIO_DIR, f"fleurs_en_{i:05d}.wav")
            sf.write(wav_path, np.array(audio["array"], dtype=np.float32), audio["sampling_rate"])
        print(f'Extracted {len(ds)} WAV files to {FLEURS_AUDIO_DIR}')
    else:
        n_wavs = len([f for f in os.listdir(FLEURS_AUDIO_DIR) if f.endswith('.wav')])
        print(f'FLEURS audio already extracted: {n_wavs} WAV files')
    print('Dataset ready!')

## Step 1: Whisper Transcription (~2-3 hrs for KenSpeech, ~1 hr for FLEURS)

- **Sw->En (KenSpeech)**: Uses KenSpeech's existing transcripts as text, runs Whisper only for word-level timestamps needed by the alignment step.
- **En->Sw (FLEURS)**: Runs Whisper normally for both transcription and timestamps.

In [ ]:
TRANSCRIPTION_DIR = f'{BASE_DIR}/transcriptions/{SOURCE_LANG}'

# Check if we can resume from a previous run
existing = len([f for f in os.listdir(TRANSCRIPTION_DIR) if f.endswith('.json')]) if os.path.exists(TRANSCRIPTION_DIR) else 0
print(f'Existing transcriptions: {existing}')

if SOURCE == 'kenspeech':
    # Sw->En: KenSpeech source (auto-downloaded, no dataset_dir needed)
    !cd {REPO_DIR} && python data/prepare/transcribe_whisper.py \
        --source kenspeech \
        --lang {SOURCE_LANG} \
        --output_dir {TRANSCRIPTION_DIR} \
        --whisper_model medium \
        --max_samples {MAX_SAMPLES} \
        --resume_from {existing}
else:
    # En->Sw: FLEURS audio directory
    !cd {REPO_DIR} && python data/prepare/transcribe_whisper.py \
        --source directory \
        --lang {SOURCE_LANG} \
        --audio_dir {FLEURS_AUDIO_DIR} \
        --output_dir {TRANSCRIPTION_DIR} \
        --whisper_model medium \
        --max_samples {MAX_SAMPLES}

In [ ]:
# Verify transcriptions
import json
from pathlib import Path

trans_files = sorted(Path(TRANSCRIPTION_DIR).glob('*.json'))
print(f'Total transcription files: {len(trans_files)}')

# Preview a sample
if trans_files:
    with open(trans_files[0]) as f:
        sample = json.load(f)
    print(f'\nSample: {trans_files[0].name}')
    print(f'  Text: {sample.get("text", "")[:100]}')
    print(f'  Duration: {sample.get("audio_duration", 0):.1f}s')
    print(f'  Words: {len(sample.get("words", []))}')
    if sample.get('words'):
        print(f'  First word: {sample["words"][0]}')

## Step 2: MADLAD Translation (~1-2 hrs)

Translate all transcriptions to the target language using MADLAD-400-3B.

In [ ]:
DIRECTION = f'{SOURCE_LANG}2{TARGET_LANG}'
TRANSLATION_DIR = f'{BASE_DIR}/translations/{DIRECTION}'

existing = len([f for f in os.listdir(TRANSLATION_DIR) if f.endswith('.json')]) if os.path.exists(TRANSLATION_DIR) else 0
print(f'Existing translations: {existing}')

!cd {REPO_DIR} && python data/prepare/translate_madlad.py \
    --input_dir {TRANSCRIPTION_DIR} \
    --output_dir {TRANSLATION_DIR} \
    --source_lang {SOURCE_LANG} \
    --target_lang {TARGET_LANG} \
    --batch_size 32 \
    --max_samples {MAX_SAMPLES} \
    --resume_from {existing}

In [ ]:
# Verify translations
trans_files = sorted(Path(TRANSLATION_DIR).glob('*.json'))
print(f'Total translation files: {len(trans_files)}')

if trans_files:
    with open(trans_files[0]) as f:
        sample = json.load(f)
    print(f'\nSample: {trans_files[0].name}')
    print(f'  Source ({SOURCE_LANG}): {sample.get("source_text", "")[:100]}')
    print(f'  Target ({TARGET_LANG}): {sample.get("translated_text", "")[:100]}')

## Step 3: Contextual Alignment (~30 min)

Compute per-word alignment between source and target text using
MADLAD-400 perplexity (Hibiki paper, Section 3.2.1, eq. 6).

This determines which source word maps to which target word,
which is needed for the silence insertion step.

In [ ]:
ALIGNMENT_DIR = f'{BASE_DIR}/alignments/{DIRECTION}'

!cd {REPO_DIR} && python data/prepare/run_pipeline.py \
    --source {SOURCE} \
    --source_lang {SOURCE_LANG} \
    --target_lang {TARGET_LANG} \
    --base_dir {BASE_DIR} \
    --max_samples {MAX_SAMPLES} \
    --step align

In [ ]:
# Verify alignments
align_files = sorted(Path(ALIGNMENT_DIR).glob('*.json'))
print(f'Total alignment files: {len(align_files)}')

if align_files:
    with open(align_files[0]) as f:
        sample = json.load(f)
    print(f'\nSample: {align_files[0].name}')
    print(f'  Source: {sample.get("source_text", "")[:80]}')
    print(f'  Target: {sample.get("translated_text", "")[:80]}')
    print(f'  Alignment pairs: {len(sample.get("alignment", []))}')
    if sample.get('alignment'):
        print(f'  First 5 pairs: {sample["alignment"][:5]}')

## Step 4: TTS Synthesis + Silence Insertion (~2-3 hrs)

This is the key step that creates the synthetic parallel speech:
1. Synthesize target-language speech from translations using MMS-TTS
2. Get word timestamps on synthesized audio with Whisper
3. Apply silence insertion for causal alignment (Hibiki paper Section 3.2)

**For Sw->En:** Uses pretrained `facebook/mms-tts-eng`
**For En->Sw:** Uses pretrained `facebook/mms-tts-swh`

In [ ]:
SYNTH_DIR = f'{BASE_DIR}/synthetic_audio/{DIRECTION}'

cmd = (
    f"cd {REPO_DIR} && python data/prepare/synthesize_tts.py"
    f" --translation_dir {TRANSLATION_DIR}"
    f" --output_dir {SYNTH_DIR}"
    f" --target_lang {TARGET_LANG}"
    f" --alignment_dir {ALIGNMENT_DIR}"
    f" --whisper_model medium"
    f" --target_sr 24000"
    f" --max_samples {MAX_SAMPLES}"
)

# No VITS model -- use pretrained MMS-TTS for both directions

print(f'Running synthesis...\n{cmd}\n')
!{cmd}

In [ ]:
# Verify synthesized audio
import IPython.display as ipd
import torchaudio

aligned_dir = f'{SYNTH_DIR}/aligned_audio'
raw_dir = f'{SYNTH_DIR}/raw_synthesis/wavs'

# Check which directory has output
wav_dir = aligned_dir if os.path.exists(aligned_dir) else raw_dir
wav_files = sorted(Path(wav_dir).glob('*.wav'))[:5]

print(f'Synthesized WAVs in {wav_dir}: {len(list(Path(wav_dir).glob("*.wav")))} total')
print()

for wav_path in wav_files:
    waveform, sr = torchaudio.load(str(wav_path))
    dur = waveform.shape[1] / sr
    print(f'{wav_path.name} ({dur:.2f}s, {sr}Hz)')
    ipd.display(ipd.Audio(waveform.squeeze().numpy(), rate=sr))

## Step 5: Encode Audio with Mimi Codec

Encode both source audio and synthesized target audio through the Mimi
neural codec to get discrete token representations.

Output: `.npy` files of shape `(Q=8, T)` -- 8 codebooks x T timesteps.

In [ ]:
AUDIO_TOKEN_DIR = f'{BASE_DIR}/audio_tokens'

# Encode source audio
if SOURCE == 'kenspeech':
    print('=== Encoding source audio (KenSpeech Swahili) ===')
    !cd {REPO_DIR} && python data/prepare/encode_audio.py \
        --source kenspeech \
        --output_dir {AUDIO_TOKEN_DIR}/kenspeech_sw \
        --num_codebooks 8 \
        --max_samples {MAX_SAMPLES} \
        --max_duration 20.0
else:
    print('=== Encoding source audio (FLEURS English) ===')
    !cd {REPO_DIR} && python data/prepare/encode_audio.py \
        --source directory \
        --audio_dir {FLEURS_AUDIO_DIR} \
        --output_dir {AUDIO_TOKEN_DIR}/fleurs_en \
        --num_codebooks 8 \
        --max_duration 20.0

In [ ]:
# Encode synthesized target audio
synth_wav_dir = f'{SYNTH_DIR}/aligned_audio'
if not os.path.exists(synth_wav_dir):
    synth_wav_dir = f'{SYNTH_DIR}/raw_synthesis/wavs'

print(f'=== Encoding synthesized {TARGET_LANG} audio ===')
!cd {REPO_DIR} && python data/prepare/encode_audio.py \
    --source directory \
    --audio_dir {synth_wav_dir} \
    --output_dir {AUDIO_TOKEN_DIR}/synth_{TARGET_LANG} \
    --num_codebooks 8 \
    --max_duration 30.0

In [ ]:
# Verify encoded tokens
import numpy as np

for subdir in [f'cv_{SOURCE_LANG}', f'synth_{TARGET_LANG}']:
    token_dir = f'{AUDIO_TOKEN_DIR}/{subdir}'
    if os.path.exists(token_dir):
        npy_files = list(Path(token_dir).glob('*.npy'))
        print(f'\n{subdir}: {len(npy_files)} token files')
        if npy_files:
            sample = np.load(str(npy_files[0]))
            print(f'  Sample shape: {sample.shape} (Q={sample.shape[0]}, T={sample.shape[1]})')
            print(f'  Duration: ~{sample.shape[1] / 12.5:.1f}s at 12.5Hz')

## Step 6: Pipeline Summary

Check all outputs and prepare for training.

In [ ]:
print('=' * 60)
print(f'PIPELINE COMPLETE: {SOURCE_LANG} -> {TARGET_LANG}')
print('=' * 60)

artifacts = {
    'Transcriptions': f'{BASE_DIR}/transcriptions/{SOURCE_LANG}',
    'Translations': f'{BASE_DIR}/translations/{DIRECTION}',
    'Alignments': f'{BASE_DIR}/alignments/{DIRECTION}',
    'Synthesized audio': SYNTH_DIR,
    f'Source tokens': f'{AUDIO_TOKEN_DIR}/{"kenspeech_sw" if SOURCE == "kenspeech" else "fleurs_en"}',
    f'Target tokens (synth_{TARGET_LANG})': f'{AUDIO_TOKEN_DIR}/synth_{TARGET_LANG}',
}

for name, path in artifacts.items():
    if os.path.exists(path):
        n_json = len(list(Path(path).rglob('*.json')))
        n_wav = len(list(Path(path).rglob('*.wav')))
        n_npy = len(list(Path(path).rglob('*.npy')))
        parts = []
        if n_json: parts.append(f'{n_json} json')
        if n_wav: parts.append(f'{n_wav} wav')
        if n_npy: parts.append(f'{n_npy} npy')
        print(f'  {name}: {", ".join(parts)}')
    else:
        print(f'  {name}: [NOT FOUND]')

print(f'\n=== Next Steps ===')
if SOURCE_LANG == 'sw':
    print('1. Re-run this notebook with SOURCE_LANG="en", TARGET_LANG="sw"')
else:
    print('1. Both directions are done!')
print('2. Run notebook 00_data_preparation.ipynb for tokenizer + text tokens')
print('3. Create S2ST manifest with create_s2st_manifest.py')
print('4. Start Stages 1-2 training on Kaggle')